# Banking Analytics EDA

This notebook performs exploratory data analysis on transaction-level, customer-level, and segment-level banking analytics datasets.

In [ ]:
# 1) Load datasets
import pandas as pd
import matplotlib.pyplot as plt

# Load transaction dataset with fallback
try:
    tx = pd.read_csv('staging_transactions.csv')
    tx_source = 'staging_transactions.csv'
except FileNotFoundError:
    tx = pd.read_csv('fact_transaction.csv')
    tx_source = 'fact_transaction.csv'

# Load customer datasets
customer_features = pd.read_csv('mart_customer_features.csv')
customer_segments = pd.read_csv('mart_customer_segments.csv')

# Resolve transaction type column name
if 'type' in tx.columns:
    tx_type_col = 'type'
elif 'transaction_type' in tx.columns:
    tx_type_col = 'transaction_type'
else:
    raise KeyError('Transaction type column not found (expected type or transaction_type).')

# Validate fraud column
if 'isFraud' not in tx.columns:
    raise KeyError('Fraud column isFraud not found in transaction dataset.')

# Ensure monetary is present in segment table
if 'monetary' not in customer_segments.columns and {'customer_id', 'monetary'}.issubset(customer_features.columns):
    customer_segments = customer_segments.merge(
        customer_features[['customer_id', 'monetary']],
        on='customer_id',
        how='left'
    )

print(f'Transaction source: {tx_source}')
print(f'Transactions: {len(tx):,}')
print(f'Customer features: {len(customer_features):,}')
print(f'Customer segments: {len(customer_segments):,}')

## Transaction Analysis

In [ ]:
# Transaction amount distribution (capped at 99th percentile)
tx_amount = tx['amount'].dropna()
amount_cap = tx_amount.quantile(0.99)

plt.figure(figsize=(10, 5))
tx_amount.clip(upper=amount_cap).hist(bins=60)
plt.title('Transaction Amount Distribution (Capped at 99th Percentile)')
plt.xlabel('Amount')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Transaction type counts
type_counts = tx[tx_type_col].value_counts(dropna=False).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
type_counts.plot(kind='bar')
plt.title('Transaction Type Counts')
plt.xlabel('Transaction Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

type_counts

In [ ]:
# Fraud vs non-fraud distribution
fraud_counts = tx['isFraud'].value_counts().sort_index()
fraud_counts.index = fraud_counts.index.map({0: 'Non-Fraud', 1: 'Fraud'})

plt.figure(figsize=(6, 4))
fraud_counts.plot(kind='bar')
plt.title('Fraud vs Non-Fraud Transactions')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

fraud_pct = tx['isFraud'].mean() * 100
print(f'Fraud transaction %: {fraud_pct:.4f}%')

## Customer Behavior Analysis

In [ ]:
# Customer frequency distribution
freq = customer_features['frequency'].dropna()
freq_cap = freq.quantile(0.99)

plt.figure(figsize=(10, 5))
freq.clip(upper=freq_cap).hist(bins=60)
plt.title('Customer Frequency Distribution (Capped at 99th Percentile)')
plt.xlabel('Frequency')
plt.ylabel('Customers')
plt.tight_layout()
plt.show()

In [ ]:
# Customer monetary distribution
monetary = customer_features['monetary'].dropna()
monetary_cap = monetary.quantile(0.99)

plt.figure(figsize=(10, 5))
monetary.clip(upper=monetary_cap).hist(bins=60)
plt.title('Customer Monetary Distribution (Capped at 99th Percentile)')
plt.xlabel('Monetary')
plt.ylabel('Customers')
plt.tight_layout()
plt.show()

In [ ]:
# Customer recency distribution
recency = customer_features['recency_days'].dropna()

plt.figure(figsize=(10, 5))
recency.hist(bins=60)
plt.title('Customer Recency Distribution')
plt.xlabel('Recency (days)')
plt.ylabel('Customers')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix for recency, frequency, monetary
corr_cols = ['recency_days', 'frequency', 'monetary']
corr_matrix = customer_features[corr_cols].corr()

print('Correlation matrix:')
print(corr_matrix.round(4))

In [ ]:
# Scatter plot: frequency vs monetary
scatter_df = customer_features[['frequency', 'monetary']].dropna()
if len(scatter_df) > 200000:
    scatter_df = scatter_df.sample(200000, random_state=42)

plt.figure(figsize=(8, 5))
plt.scatter(scatter_df['frequency'], scatter_df['monetary'], alpha=0.3, s=8)
plt.xlabel('Frequency')
plt.ylabel('Monetary')
plt.title('Frequency vs Monetary')
plt.tight_layout()
plt.show()

## Segmentation Analysis

In [ ]:
# Segment distribution
segment_counts = customer_segments['segment_label'].value_counts(dropna=False).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
segment_counts.plot(kind='bar')
plt.title('Customer Segment Distribution')
plt.xlabel('Segment')
plt.ylabel('Customers')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

segment_counts

In [ ]:
# Revenue contribution by segment
segment_revenue = (
    customer_segments.groupby('segment_label', dropna=False, as_index=False)['monetary']
    .sum()
    .sort_values('monetary', ascending=False)
)
segment_revenue['revenue_pct'] = segment_revenue['monetary'] / segment_revenue['monetary'].sum() * 100

plt.figure(figsize=(10, 5))
plt.bar(segment_revenue['segment_label'], segment_revenue['revenue_pct'])
plt.title('Revenue Contribution by Segment')
plt.xlabel('Segment')
plt.ylabel('Revenue Contribution (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

segment_revenue

In [ ]:
# Risk band distribution
risk_counts = customer_segments['risk_band'].value_counts(dropna=False).sort_values(ascending=False)

plt.figure(figsize=(6, 4))
risk_counts.plot(kind='bar')
plt.title('Risk Band Distribution')
plt.xlabel('Risk Band')
plt.ylabel('Customers')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

risk_counts

## Key Business Insights

In [ ]:
# Final business insights
total_customers = len(customer_features)
total_value = customer_features['monetary'].sum()

# Revenue concentration (top 20%)
top_20_n = max(1, int(total_customers * 0.20))
top20_value_share = customer_features.nlargest(top_20_n, 'monetary')['monetary'].sum() / total_value * 100

# Dormant customer %
dormant_mask = (customer_features['recency_days'] > 90) & (customer_features['txns_30d'] == 0)
dormant_pct = dormant_mask.mean() * 100

# Risk distribution
high_risk_pct = (customer_segments['risk_band'] == 'HIGH').mean() * 100
medium_risk_pct = (customer_segments['risk_band'] == 'MEDIUM').mean() * 100

# Fraud % at transaction level
fraud_pct = tx['isFraud'].mean() * 100

# Largest segment
largest_segment = customer_segments['segment_label'].value_counts().idxmax()
largest_segment_pct = customer_segments['segment_label'].value_counts(normalize=True).max() * 100

print(f'Revenue concentration (Top 20%): {top20_value_share:.2f}%')
print(f'Dormant customer %: {dormant_pct:.2f}%')
print(f'Risk distribution -> HIGH: {high_risk_pct:.2f}%, MEDIUM: {medium_risk_pct:.2f}%')
print(f'Fraud % (transactions): {fraud_pct:.4f}%')
print(f'Largest segment: {largest_segment} ({largest_segment_pct:.2f}%)')